# Phase 4 — Pipeline Complet (A1 → A7)
**YouTube Quality Analyzer** | Phase 4 — Intégration

Ce notebook exécute le pipeline LangGraph complet sans CSV réel :
- Injection de commentaires synthétiques (bypass A1)
- Exécution de A2 → A3 ‖ A4 ‖ A5 → A6 → A7
- Inspection du rapport final
- Comparaison avec/sans topic (A7)
- Vérification des contraintes PRD (poids immuables, quality_label)

> Les LLM sont **mockés** — pipeline testable hors Kaggle.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from unittest.mock import MagicMock, patch
from graph import run_pipeline

## 1. Commentaires synthétiques (bypass CSV)

In [ ]:
RAW_COMMENTS = [
    {"text": "This is a great tutorial on machine learning!", "video_id": "ml_vid_001"},
    {"text": "Very informative, learned a lot about gradient descent.", "video_id": "ml_vid_001"},
    {"text": "The approach mirrors Goodfellow ch.9 — excellent depth.", "video_id": "ml_vid_001"},
    {"text": "Best ML course I have taken, clearly explained backprop.", "video_id": "ml_vid_001"},
    {"text": "Could you cover transformer architectures in the next video?", "video_id": "ml_vid_001"},
    {"text": "lol 😂", "video_id": "ml_vid_001"},
    {"text": "SUBSCRIBE TO MY CHANNEL: bit.ly/spam", "video_id": "ml_vid_001"},
]

VIDEO_ID = "ml_vid_001"
TOPIC    = "machine learning"
print(f"Corpus : {len(RAW_COMMENTS)} commentaires | Vidéo : {VIDEO_ID} | Thème : {TOPIC}")

## 2. Exécution pipeline — tous LLM mockés

In [ ]:
def _make_mock_llm():
    mock = MagicMock()
    def _invoke(messages, **kw):
        prompt = str(messages[-1].content if hasattr(messages[-1], "content") else messages)
        if "sentiment_label" in prompt:
            body = json.dumps({"sentiment_label": "positive", "sentiment_score": 0.78, "rationale": "Positive corpus"})
        elif "informativeness" in prompt:
            body = json.dumps({"informativeness": 0.75, "argumentation": 0.65, "constructiveness": 0.80, "high_quality_indices": [0, 2, 3], "rationale": "Good discourse"})
        elif "noise_ratio" in prompt:
            body = json.dumps({"spam_ratio": 0.10, "offtopic_ratio": 0.05, "reaction_ratio": 0.10, "toxic_ratio": 0.0, "bot_ratio": 0.0, "noise_ratio": 0.20, "rationale": "Low noise"})
        elif "pertinence_score" in prompt:
            body = json.dumps({"pertinence_score": 0.85, "verdict": "The video is highly relevant to machine learning."})
        else:
            body = "The comment section reflects quality engagement with the topic."
        return MagicMock(content=body)
    mock.invoke.side_effect = _invoke
    return mock

patches = [
    patch("agents.a3_sentiment.get_llm", return_value=_make_mock_llm()),
    patch("agents.a4_discourse.get_llm",  return_value=_make_mock_llm()),
    patch("agents.a5_noise.get_llm",      return_value=_make_mock_llm()),
    patch("agents.a6_synthesizer.get_llm",return_value=_make_mock_llm()),
    patch("agents.a7_topic_matcher.get_llm", return_value=_make_mock_llm()),
]
for p in patches:
    p.start()

try:
    report = run_pipeline(raw_comments=RAW_COMMENTS, video_id=VIDEO_ID, topic=TOPIC)
finally:
    for p in patches:
        p.stop()

print("=" * 60)
print(f"  Score_Sentiment : {report.get('details', {}).get('sentiment', {}).get('sentiment_score')}")
print(f"  Score_Discours  : {report.get('details', {}).get('discourse', {}).get('discourse_score')}")
print(f"  Score_Bruit     : {report.get('details', {}).get('noise', {}).get('noise_score')}")
print(f"  Score_Global    : {report.get('score_global')}")
print(f"  Score_Pertinence: {report.get('score_pertinence')}")
print(f"  Score_Final     : {report.get('score_final')}")
print(f"  Quality Label   : {report.get('quality_label')}")
print(f"  Topic Verdict   : {report.get('topic_verdict')}")
print(f"  Erreurs         : {report.get('errors')}")
print("=" * 60)

## 3. Sans topic — Score_Final = Score_Global

In [ ]:
with patch("agents.a3_sentiment.get_llm", return_value=None), \
     patch("agents.a4_discourse.get_llm",  return_value=None), \
     patch("agents.a5_noise.get_llm",      return_value=None), \
     patch("agents.a6_synthesizer.get_llm",return_value=None), \
     patch("agents.a7_topic_matcher.get_llm", return_value=None):
    report_no_topic = run_pipeline(raw_comments=RAW_COMMENTS, video_id=VIDEO_ID, topic="")

print("Sans topic :")
print(f"  Score_Global : {report_no_topic.get('score_global')}")
print(f"  Score_Final  : {report_no_topic.get('score_final')}")
assert report_no_topic["score_global"] == report_no_topic["score_final"], \
    "Sans topic, score_final doit être égal à score_global"
print("  Assertion OK : score_final == score_global quand topic est absent.")

## Résumé Pipeline Complet

| Étape | Statut |
|---|---|
| A1 → A2 → A3‖A4‖A5 → A6 → A7 (avec LLM mockés) | Confirmé |
| Score_Global ∈ [0, 100] | Confirmé |
| Score_Final ∈ [0, 100] | Confirmé |
| Sans topic : Score_Final = Score_Global | Confirmé |
| Quality Label valide | Confirmé |
| Pipeline sans LLM (fallbacks) | Confirmé |

> **Prochaine étape** : fournir `data/raw/comments.csv` → exécuter sur données réelles → gold standard → tag `v0.1.0`

**Formules PRD §4.3 :**
```
Score_Global = 0.35 × Score_Sentiment + 0.40 × Score_Discours + 0.25 × Score_Bruit
Score_Final  = 0.60 × Score_Global    + 0.40 × Score_Pertinence
```